# Dialogue Log Verification System

    "This notebook verifies and selects the best dialogue from Gemini-3.1 and GPT-5.2 generated seeds using Claude Sonnet 4.5 as an independent arbiter with binary pass/fail validation on 5 criteria.\n",

## Execution Flow

The cells execute in 7 sequential steps:
1. Import libraries and set file paths
2. Load Gemini-3.1 and GPT-5.2 CSV datasets
3. Review binary validation system and decision logic
4. Initialize Claude Sonnet 4.5 verifier and define verification function
5. Run verification loop (currently limited to first 2 games for testing)
6. Create output DataFrames from verification results with per-criterion tracking
7. Save verified results to CSV files

## Output Files

1. **Datasets/verified/verified_r1_seeds_combined.csv**: Selected dialogues with source labels
2. **Datasets/verified/verified_r1_criteria_scores.csv**: Detailed per-criterion pass/fail tracking for analysis

## Step 1: Import Libraries

Import required Python libraries and set file paths.

In [ ]:
import sys
from pathlib import Path

# Locate AVCORP/ so paths work from the notebook folder or the repo root
_cwd = Path.cwd().resolve()
AVCORP_DIR = next(d for p in [_cwd, *_cwd.parents] for d in (p, p / "AVCORP")
                  if (d / "paths.py").is_file())
sys.path[:0] = [str(AVCORP_DIR), str(AVCORP_DIR / "common")]
from paths import TACTICS_KB, SEEDS_DIR, VERIFIED_DIR

In [1]:
# Import Required Libraries
import pandas as pd
import json
import os
import re
from typing import Dict, Tuple, Optional
from anthropic import Anthropic
import time

# Initialize paths
DATA_DIR = "."
GEMINI3_FILE = SEEDS_DIR / "generated_r1_seeds_gemini3.csv"
GPT52_FILE = SEEDS_DIR / "generated_r1_seeds_gpt5_2.csv"
OUTPUT_FILE_COMBINED = VERIFIED_DIR / "verified_r1_seeds_combined.csv"
OUTPUT_FILE_CRITERIA = VERIFIED_DIR / "verified_r1_criteria_scores.csv"

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Load Datasets

Load the Gemini-3.1 and GPT-5.2 generated seed datasets.

In [2]:
# Load Generated Seeds from GPT Models
gemini3_df = pd.read_csv(GEMINI3_FILE)
gpt52_df = pd.read_csv(GPT52_FILE)

print(f"Gemini-3.1 dataset loaded: {gemini3_df.shape[0]} rows, {gemini3_df.shape[1]} columns")
print(f"GPT-5.2 dataset loaded: {gpt52_df.shape[0]} rows, {gpt52_df.shape[1]} columns")
print(f"\nColumns in Gemini-3.1: {list(gemini3_df.columns)}")
print(f"Sample row (first game):")
print(f"  game_id: {gemini3_df.iloc[0]['game_id']}")
print(f"  player_roles: {gemini3_df.iloc[0]['player_roles'][:80]}...")
print(f"  matrix_tactic_scale sample: {gemini3_df.iloc[0]['matrix_tactic_scale'][:100]}...")

Gemini-3.1 dataset loaded: 225 rows, 9 columns
GPT-5.2 dataset loaded: 225 rows, 9 columns

Columns in Gemini-3.1: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale']
Sample row (first game):
  game_id: G026
  player_roles: {"P1":"Good",
  "P2":"Good",
  "P3":"Evil",
  "P4":"Evil",
  "P5":"Good"}...
  matrix_tactic_scale sample: {
  "P2": {"row":"Counterfactual","col":"Cooperative","tactic":"Sincere false belief","scale":"GRS",...


## Step 3: Binary Multi-Criteria Validation System

**Validation Criteria:**

Claude Sonnet 4.5 evaluates each dialogue using **binary pass/fail checks** on 5 criteria:

1. **Discussion Coherence & Tactic Situational Fit** (1/0) - Each player's tactic makes situational sense given the social dynamics of the discussion
2. **Public History Alignment** (1/0) - Dialogue contextually appropriate to game state
3. **Tactic-Dialogue Alignment** (1/0) - Annotation accurately describes what was said, with correct taxonomy, scale, and alignment
4. **Authenticity & Skill Consistency** (1/0) - Natural dialogue with skill-modulated sophistication
5. **Dialogue Format Compliance** (1/0) - Starts with proper header, exactly 4 non-protagonist speakers, each speaks once

**Decision Logic:**

1. **Both 5/5 pass** → Claude Sonnet 4.5 performs pairwise comparison to break tie
2. **One 5/5, other <5/5** → Select the 5/5 response
3. **At least one 4/5**:
   - One 4/5, other <4/5 → Choose 4/5, **Targeted Correction** (fix only the failing criterion)
   - Both 4/5 → Choose GPT-5.2, **Targeted Correction**
4. **Both ≤3/5** → **Full Custom Generation** (generate from scratch)

## Step 3a: Initialize Claude Sonnet 4.5

Set up the Anthropic client and system prompt.

In [3]:
# Initialize Claude Sonnet 4.5 Verifier
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'), max_retries=5)

SYSTEM_PROMPT = """You are an expert verifier for the Avalon social deception game with deep knowledge of deception tactics and game mechanics. Your task is to evaluate pairs of AI-generated dialogue responses, assess their quality, and determine the best outcome: select the superior response as-is, apply targeted corrections to fix specific issues in the better response, or generate an entirely new dialogue with appropriate tactics and scales if both responses are inadequate."""

# ── Build compact tactic taxonomy reference from knowledge base ───────────────
with open(TACTICS_KB, 'r') as _f:
    _kb = json.load(_f)

_COLOR_LABEL = {'green': 'Good-only', 'red': 'Evil-only', 'blue': 'Both alignments'}
_COLOR_ORDER = ['green', 'red', 'blue']
_grouped = {c: [] for c in _COLOR_ORDER}
for cell_key, cell in _kb['behavior_matrix'].items():
    row, col = cell_key.split('|')
    for tactic in cell['tactics']:
        _grouped[cell['color']].append((row, col, tactic))

_scale_lines = []
for scale_key, sd in _kb['scale_definitions'].items():
    level_strs = '; '.join(f'{lvl}: {desc}' for lvl, desc in sd['levels'].items())
    _scale_lines.append(f'  {scale_key} ({sd["name"]}) — for {sd["for"]}: {level_strs}')

_tax_lines = ['TACTIC ANNOTATION TAXONOMY']
_tax_lines.append('')
_tax_lines.append('The 4×4 Behavioral Matrix has two dimensions:')
_tax_lines.append('  Row (Information Strategy): Transparent | Selective/Framing | Careless-to-truth | Counterfactual')
_tax_lines.append('    - Transparent: speaker openly shares clear evidence-based information')
_tax_lines.append('    - Selective/Framing: speaker shapes communication through selective disclosure, emphasis, or framing')
_tax_lines.append('    - Careless-to-truth: speaker is indifferent to factual accuracy, i.e., bullshitting, vague speculation, or noise')
_tax_lines.append('    - Counterfactual: speaker asserts or believes information contrary to reality, i.e., honest mistake, self-deception, or hard lying')
_tax_lines.append('  Column (Goal Orientation): Cooperative | Defensive | Opportunistic | Adversarial')
_tax_lines.append('    - Cooperative: helps the group reach accurate conclusions')
_tax_lines.append('    - Defensive: protects self from suspicion or blame')
_tax_lines.append('    - Opportunistic: exploits the social moment for personal or team advantage')
_tax_lines.append('    - Adversarial: actively undermines specific players or misdirects the group')
_tax_lines.append('')
_tax_lines.append('Valid tactic names by alignment (use ONLY these exact names):')
for color in _COLOR_ORDER:
    _tax_lines.append(f'  {_COLOR_LABEL[color]}:')
    for row, col, tactic in _grouped[color]:
        _tax_lines.append(f'    [{row} | {col}] {tactic}')
_tax_lines.append('')
_tax_lines.append('Skill scales ("scale" field in matrix_tactic_scale):')
_tax_lines.extend(_scale_lines)
_tax_lines.append('  Note: Good players at Moderate/Low skill may legitimately use Blue (Both-alignment) tactics.')
_tax_lines.append('  Do NOT penalize a Good player for using Deflection, Self-deception, Face-saving blather,')
_tax_lines.append('  Vague filler, Noisy sincerity, Self-justification, Pragmatic silence, Omission to save face,')
_tax_lines.append('  Half-truths, or Rationalized error — these are valid defensive behaviors for Good players.')

TACTIC_TAXONOMY_REF = '\n'.join(_tax_lines)
print("\n" + TACTIC_TAXONOMY_REF + "\n")

print("✓ Claude Sonnet 4.5 system prompt initialized")
print(f"✓ Tactic taxonomy reference built: {len(_grouped['green']) + len(_grouped['red']) + len(_grouped['blue'])} tactics across 3 alignment groups")


TACTIC ANNOTATION TAXONOMY

The 4×4 Behavioral Matrix has two dimensions:
  Row (Information Strategy): Transparent | Selective/Framing | Careless-to-truth | Counterfactual
    - Transparent: speaker openly shares clear evidence-based information
    - Selective/Framing: speaker shapes communication through selective disclosure, emphasis, or framing
    - Careless-to-truth: speaker is indifferent to factual accuracy, i.e., bullshitting, vague speculation, or noise
    - Counterfactual: speaker asserts or believes information contrary to reality, i.e., honest mistake, self-deception, or hard lying
  Column (Goal Orientation): Cooperative | Defensive | Opportunistic | Adversarial
    - Cooperative: helps the group reach accurate conclusions
    - Defensive: protects self from suspicion or blame
    - Opportunistic: exploits the social moment for personal or team advantage
    - Adversarial: actively undermines specific players or misdirects the group

Valid tactic names by alignment (us

## Step 3b: Define Verification Function

Create the function that compares and evaluates dialogue pairs.

In [4]:

def reorder_matrix_by_speaking_order(dialogue: str, matrix_input) -> str:
    """Reorder matrix_tactic_scale keys to match speaking order in the dialogue."""
    try:
        speakers = re.findall(r'(?m)^(P\d+):', dialogue or '')
        seen, ordered_speakers = set(), []
        for s in speakers:
            if s not in seen:
                seen.add(s)
                ordered_speakers.append(s)
        matrix = json.loads(matrix_input) if isinstance(matrix_input, str) else (matrix_input or {})
        reordered = {s: matrix[s] for s in ordered_speakers if s in matrix}
        for k, v in matrix.items():
            if k not in reordered:
                reordered[k] = v
        lines = [f'  "{k}": ' + json.dumps(v, separators=(',', ':')) for k, v in reordered.items()]
        return '{\n' + ',\n'.join(lines) + '\n}'
    except Exception:
        if isinstance(matrix_input, str):
            return matrix_input
        m = matrix_input or {}
        lines = [f'  "{k}": ' + json.dumps(v, separators=(',', ':')) for k, v in m.items()]
        return '{\n' + ',\n'.join(lines) + '\n}'


def verify_dialogue_pair(game_id: str,
                         round_id: str,
                         player_roles: str,
                         protagonist: str,
                         public_history: str,
                         dialogue_gemini3: str,
                         tactic_scale_gemini3: str,
                         dialogue_gpt52: str,
                         tactic_scale_gpt52: str) -> Dict:
    """
    Verify both dialogue responses using Claude Sonnet 4.5 with binary criteria validation.
    
    Returns: {
        'gemini3_criteria': {'coherence': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gemini3_total': int (0-5),
        'gpt52_criteria': {'coherence': bool, 'history': bool, 'matrix': bool, 'authenticity': bool, 'format': bool},
        'gpt52_total': int (0-5),
        'recommendation': str,
        'selected_dialogue': str,
        'selected_tactic_scale': str,
        'correction_mode': str (None, Targeted, Full_Custom, Pairwise_Tiebreaker),
        'failed_criterion': str or None,
        'reasoning': str,
        'needs_retry': bool  — True when tc_dialogue or cd_dialogue was None/empty (silent fallback applied)
    }
    """
    
    verification_prompt = f"""You are evaluating two AI-generated Avalon Round 1 discussion logs to determine which is superior. Both responses were generated for the same game context. Your task is to validate both using BINARY PASS/FAIL checks on 5 criteria, then recommend the best outcome: select one response as-is, apply a targeted correction to the better response, or generate a completely new dialogue with appropriate tactics and scales if both are inadequate.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Public History (up to current round):
{public_history}

---

**RESPONSE A:**

Discussion Log:
{dialogue_gemini3}

Matrix Tactic Scale:
{tactic_scale_gemini3}

---

**RESPONSE B:**

Discussion Log:
{dialogue_gpt52}

Matrix Tactic Scale:
{tactic_scale_gpt52}

---

**TACTIC ANNOTATION FRAMEWORK:**
Both responses were generated from identical pre-assigned tactic plans. The Matrix Tactic Scale shown for each response is that model's annotation of its own dialogue. Neither is canonical — treat them as starting references, not ground truth.

{TACTIC_TAXONOMY_REF}

Each speaker's annotation in matrix_tactic_scale must use this structure:
  "PX": {{"row": "<Information Strategy>", "col": "<Goal Orientation>", "tactic": "<exact name from taxonomy>", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Discussion Coherence & Tactic Situational Fit**:
   1 = Each player's tactic makes situational sense given the social dynamics of the discussion — their conversational position, what other players are saying, and how tensions develop across the exchange
   0 = A player's tactic is socially incongruous given the discussion flow (e.g., "Pragmatic silence" for a player being directly accused by multiple others; "Vague filler" for a player everyone is turning to for information). The utterance may match the label literally, but the tactic choice is implausible for that social moment.

2. **Public History Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns
   0 = Ignores prior events or contradicts established game state

3. **Tactic-Dialogue Alignment**:
   1 = Each player's annotation accurately describes what they actually said: the tactic name is a valid entry from the taxonomy, the row/col match the tactic's position in the 4×4 matrix, scale is GRS for Good players and Mach-IV for Evil players, and the level (High/Moderate/Low) is plausible given the sophistication of the utterance
   0 = The annotation mislabels what the player actually said (e.g., "Pragmatic silence" for a player who gave a 4-sentence analysis; "Hard lying" for a player who spoke honestly), OR a tactic name is not in the valid taxonomy, OR scale/row/col are inconsistent with the tactic definition
   Framing: the dialogue is the ground truth — the annotation must describe what was written, not what was planned.

4. **Authenticity & Skill Consistency**:
   1 = The dialogue reads as a believable Avalon exchange with natural accusation/defense patterns, and the linguistic sophistication of each utterance is consistent with the speaker's assigned skill level.
   0 = The dialogue feels stilted or artificial, or the sophistication of an utterance clearly mismatches the assigned level (e.g., elaborate strategic reasoning from a Low-skill player, or uncertain hedging from a High-skill player).

5. **Dialogue Format Compliance**:
   1 = Starts with "Discussion after Quest X:", exactly 4 speakers (all non-protagonists), each speaks exactly once, in this format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong or missing header, incorrect speaker count, protagonist appears as speaker, or any format violations

**DECISION LOGIC:**

Apply this logic based on how many criteria each response passes:

1. **Both responses 5/5** → Use PAIRWISE_TIEBREAKER
   - Compare both dialogues to determine which is slightly better overall
   - Output RECOMMENDATION: PAIRWISE_TIEBREAKER

2. **One response 5/5, other <5/5** → Choose the 5/5 response
   - Output RECOMMENDATION: RESPONSE_A (if Response A is 5/5)
   - Output RECOMMENDATION: RESPONSE_B (if Response B is 5/5)

3. **At least one response 4/5** → Apply targeted correction
   - If only one is 4/5: choose that one → CORRECTION_A or CORRECTION_B
   - If both are 4/5: compare both holistically (same pairwise judgment as rule 1 — which has stronger authenticity, history alignment, and tactic fit overall?) and choose the stronger base → CORRECTION_A or CORRECTION_B
   - Output RECOMMENDATION: CORRECTION_A or CORRECTION_B
   - The targeted_correction must correct the CHOSEN response (A or B)

4. **Both responses ≤3/5** → Generate custom dialogue
   - Output RECOMMENDATION: CUSTOM_GENERATION

**EVALUATION PROCESS:**

1. Evaluate RESPONSE A against all 5 criteria using binary PASS/FAIL checks
2. Evaluate RESPONSE B against all 5 criteria using the same binary checks
3. Count total passes for each response (X/5)
4. Apply the DECISION LOGIC rules based on the pass counts
5. Generate output in the exact format below

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object with the following structure:

```json
{{
  "response_a": {{
    "criteria": {{
      "coherence": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "response_b": {{
    "criteria": {{
      "coherence": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "history": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "matrix": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "authenticity": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}},
      "format": {{"score": 1/0, "explanation": "Brief explanation why it passed/failed"}}
    }},
    "total": x
  }},
  "recommendation": "RESPONSE_A / RESPONSE_B / PAIRWISE_TIEBREAKER / CORRECTION_A / CORRECTION_B / CUSTOM_GENERATION",
  "reasoning": "2-3 sentences explaining the decision",
  "pairwise_winner": "RESPONSE_A / RESPONSE_B / null",
  "pairwise_reasoning": "1-2 sentences explaining pairwise selection / null",
  "failed_criterion": "Coherence / History / Matrix / Authenticity / Format / null",
  "targeted_correction": {{
    "dialogue": "Full corrected dialogue text",
    "matrix_tactic_scale": {{"P2": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }},
  "custom_dialogue": {{
    "dialogue": "Complete new dialogue text",
    "matrix_tactic_scale": {{"P2": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }}
}}
```

**Field Explanations:**
- **score**: 1 = PASS, 0 = FAIL
- **explanation**: Brief explanation required for BOTH pass and fail to justify the score
- **total**: Integer count of passed criteria (0-5), NOT a fraction string
- **recommendation**: One of: "RESPONSE_A", "RESPONSE_B", "PAIRWISE_TIEBREAKER", "CORRECTION_A", "CORRECTION_B", "CUSTOM_GENERATION"
- **pairwise_winner**: Only if recommendation is PAIRWISE_TIEBREAKER (both 5/5), set to "RESPONSE_A" or "RESPONSE_B"
- **failed_criterion**: Only if recommendation is CORRECTION_A or CORRECTION_B, set to the failing criterion of the CHOSEN response: "Coherence", "History", "Matrix", "Authenticity", or "Format"; otherwise null
- **targeted_correction**: Corrects the CHOSEN response (matching recommendation — A or B). Only if failed_criterion is set; an object with:
  - "dialogue": full corrected dialogue text of the CHOSEN response, fixing its failed criterion
  - "matrix_tactic_scale": JSON mapping each speaker to tactic annotation; stay close to the tactic annotations shown in both matrices, updating only fields the corrected dialogue requires
  Otherwise null.
- **custom_dialogue**: Only if recommendation is CUSTOM_GENERATION (both ≤3/5); an object with:
  - "dialogue": complete new dialogue (exactly 4 speakers, protagonist excluded, "Discussion after Quest X:" format)
  - "matrix_tactic_scale": Use Response B's matrix_tactic_scale as the baseline for each speaker. Keep each speaker's tactic, row, col, scale, and level from Response B unless the new dialogue strictly requires a change. If any speaker's annotation must change, update only that speaker and ensure the revised tactic, row, col, scale, and level are valid under the tactic annotation framework and match what the new dialogue actually expresses. Generate entirely fresh dialogue content that authentically expresses those tactics in this specific game context.
  Otherwise null.
- Set **targeted_correction** to null if recommendation is not a CORRECTION
- Set **custom_dialogue** to null if recommendation is not CUSTOM_GENERATION

**IMPORTANT:** Return ONLY the JSON object, no markdown code blocks, no extra text."""
    
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": verification_prompt}
            ]
        )
        response_text = response.content[0].text
        
        # DEBUG: Print raw response to diagnose parsing issues
        # print(f"\n{'='*80}")
        # print(f"RAW CLAUDE RESPONSE FOR {game_id}:")
        # print(f"{'='*80}")
        # print(response_text)
        # print(f"{'='*80}\n")
        
        # Parse JSON response
        try:
            # Remove markdown code blocks if present
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                # Remove opening ``` line
                lines = lines[1:]
                # Find and remove closing ```
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)
            
            # Parse JSON
            data = json.loads(clean_text)
            
            # Build result dictionary from JSON
            result = {
                'gemini3_criteria': {
                    'coherence': data['response_a']['criteria']['coherence']['score'] == 1,
                    'history': data['response_a']['criteria']['history']['score'] == 1,
                    'matrix': data['response_a']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_a']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_a']['criteria']['format']['score'] == 1
                },
                'gemini3_total': int(data['response_a']['total']) if isinstance(data['response_a']['total'], int) else int(str(data['response_a']['total']).split('/')[0]),
                'gemini3_criteria_explanations': {
                    criterion: data['response_a']['criteria'][criterion].get('explanation', '')
                    for criterion in ['coherence', 'history', 'matrix', 'authenticity', 'format']
                },
                'gpt52_criteria': {
                    'coherence': data['response_b']['criteria']['coherence']['score'] == 1,
                    'history': data['response_b']['criteria']['history']['score'] == 1,
                    'matrix': data['response_b']['criteria']['matrix']['score'] == 1,
                    'authenticity': data['response_b']['criteria']['authenticity']['score'] == 1,
                    'format': data['response_b']['criteria']['format']['score'] == 1
                },
                'gpt52_total': int(data['response_b']['total']) if isinstance(data['response_b']['total'], int) else int(str(data['response_b']['total']).split('/')[0]),
                'gpt52_criteria_explanations': {
                    criterion: data['response_b']['criteria'][criterion].get('explanation', '')
                    for criterion in ['coherence', 'history', 'matrix', 'authenticity', 'format']
                },
                'recommendation': data['recommendation'],
                'reasoning': data['reasoning'],
                'pairwise_winner': data.get('pairwise_winner'),
                'pairwise_reasoning': data.get('pairwise_reasoning', ''),
                'failed_criterion': data.get('failed_criterion'),
                'targeted_correction': data.get('targeted_correction'),
                'custom_dialogue': data.get('custom_dialogue'),
                'selected_dialogue': None,
                'selected_tactic_scale': None,
                'correction_mode': None,
                'needs_retry': False,   # set True below when tc/cd silently falls back
            }
            
            # Determine final selection based on scores
            s1 = result['gemini3_total']
            s2 = result['gpt52_total']
            
            if s1 == 5 and s2 == 5:
                # Both 5/5 - use pairwise tiebreaker
                if result['pairwise_winner'] and 'RESPONSE_A' in result['pairwise_winner'].upper():
                    result['recommendation'] = 'RESPONSE_A'
                    result['selected_dialogue'] = dialogue_gemini3
                    result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gemini3, tactic_scale_gemini3)
                else:
                    result['recommendation'] = 'RESPONSE_B'
                    result['selected_dialogue'] = dialogue_gpt52
                    result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gpt52, tactic_scale_gpt52)
                result['correction_mode'] = 'Pairwise_Tiebreaker'
                
            elif s1 == 5:
                # Only Response A is 5/5
                result['recommendation'] = 'RESPONSE_A'
                result['selected_dialogue'] = dialogue_gemini3
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gemini3, tactic_scale_gemini3)
                result['correction_mode'] = None
                
            elif s2 == 5:
                # Only Response B is 5/5
                result['recommendation'] = 'RESPONSE_B'
                result['selected_dialogue'] = dialogue_gpt52
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(dialogue_gpt52, tactic_scale_gpt52)
                result['correction_mode'] = None
                
            elif s1 == 4 or s2 == 4:
                # At least one 4/5 - targeted correction
                tc          = result.get('targeted_correction') or {}
                tc_dialogue = tc.get('dialogue') if isinstance(tc, dict) else None
                tc_matrix   = tc.get('matrix_tactic_scale') if isinstance(tc, dict) else None
                
                if s1 == 4 and s2 == 4:
                    # Both 4/4 — trust Claude's pairwise judgment (same logic as 5/5 tiebreaker)
                    claude_rec = result.get('recommendation', 'CORRECTION_B')
                    if 'CORRECTION_A' in claude_rec:
                        result['recommendation'] = 'CORRECTION_A'
                        final_dialogue = tc_dialogue or dialogue_gemini3
                        final_matrix   = tc_matrix   or tactic_scale_gemini3
                    else:
                        result['recommendation'] = 'CORRECTION_B'
                        final_dialogue = tc_dialogue or dialogue_gpt52
                        final_matrix   = tc_matrix   or tactic_scale_gpt52
                elif s1 == 4:
                    result['recommendation'] = 'CORRECTION_A'
                    final_dialogue = tc_dialogue or dialogue_gemini3
                    final_matrix   = tc_matrix   or tactic_scale_gemini3
                else:  # s2 == 4
                    result['recommendation'] = 'CORRECTION_B'
                    final_dialogue = tc_dialogue or dialogue_gpt52
                    final_matrix   = tc_matrix   or tactic_scale_gpt52
                
                if tc_dialogue is None:
                    print(f"  ⚠️  WARNING: targeted_correction.dialogue is None for {game_id} "
                          f"(rec={result['recommendation']}) — falling back to uncorrected base dialogue")
                    result['needs_retry'] = True
                
                result['selected_dialogue']     = final_dialogue
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    final_dialogue,
                    json.dumps(final_matrix, separators=(',', ':')) if isinstance(final_matrix, dict) else final_matrix
                )
                result['correction_mode'] = 'Targeted'
                
            else:
                # Both ≤3/5 - full custom generation; anchor matrix to Response B's assignments
                cd          = result.get('custom_dialogue') or {}
                cd_dialogue = cd.get('dialogue') if isinstance(cd, dict) else None
                cd_matrix   = cd.get('matrix_tactic_scale') if isinstance(cd, dict) else None
                if not cd_dialogue:
                    print(f"  ⚠️  WARNING: Custom dialogue is empty for {game_id}!")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Full_Custom'
                result['selected_dialogue']     = cd_dialogue
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    cd_dialogue or '',
                    json.dumps(cd_matrix, separators=(',', ':')) if isinstance(cd_matrix, dict) else (cd_matrix or tactic_scale_gpt52)
                )
                result['recommendation'] = 'CUSTOM_GENERATION'
            return result, response_text
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            print(f"Cleaned text: {clean_text[:500]}...")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing JSON for {game_id}: {e}")
            return None, str(e)
    except Exception as e:
        print(f"Error verifying game {game_id}: {str(e)}")
        return None, str(e)


## Step 4a: Run Verification Loop

Process games and collect verification results.

In [ ]:

# Helper function to format matrix tactic scale with pretty-printing
def format_matrix_tactic_scale(json_str: str) -> str:
    """Format JSON with linebreaks for readability"""
    try:
        data = json.loads(json_str)
        # Reconstruct with pretty-printing: 2-space indent
        return json.dumps(data, indent=2)
    except:
        return json_str

def format_check(val):
    return 1 if val else 0

# Compare and Validate Dialogue Rows
# Initialize results storage
verified_results = []
criteria_results = []
verification_stats = {
    'total_games': 0,
    'response_1_selected': 0,
    'response_2_selected': 0,
    'targeted_correction_1': 0,
    'targeted_correction_2': 0,
    'pairwise_tiebreaker': 0,
    'full_custom_generation': 0,
    'errors': 0
}

# Tracks indices that need a retry in Step 4b.
# Each entry: {'idx': int, 'game_id': str, 'round_id': str/int, 'reason': str}
#   reason = 'api_error'   — verify_dialogue_pair returned None (API / parse failure)
#   reason = 'Targeted'    — tc_dialogue was None; fell back to uncorrected base
#   reason = 'Full_Custom' — cd_dialogue was empty; selected_dialogue stored as None
retry_rows = []

# Process first 2 games as a sample/test run. Replace 2 with 225 or len(gemini3_df)
TEST_LIMIT = len(gemini3_df)
print(f"Starting verification of {min(TEST_LIMIT, len(gemini3_df))} games...")
print("=" * 80)

for idx in range(min(TEST_LIMIT, len(gemini3_df))):
    game_id = gemini3_df.iloc[idx]['game_id']
    round_id = gemini3_df.iloc[idx]['round_id']
    score_id = f"{game_id}/{round_id}"
    
    # Extract fields from both datasets
    player_roles = gemini3_df.iloc[idx]['player_roles']
    protagonist = gemini3_df.iloc[idx]['role_id']  # Protagonist who doesn't speak
    public_history = gemini3_df.iloc[idx]['public_history']
    
    dialogue_gemini3 = gemini3_df.iloc[idx]['discussion_log']
    tactic_scale_gemini3 = gemini3_df.iloc[idx]['matrix_tactic_scale']
    
    dialogue_gpt52 = gpt52_df.iloc[idx]['discussion_log']
    tactic_scale_gpt52 = gpt52_df.iloc[idx]['matrix_tactic_scale']
    
    # Verify the pair
    print(f"\n[{idx+1}/{min(TEST_LIMIT, len(gemini3_df))}] Verifying game {score_id}...")
    result, raw_response = verify_dialogue_pair(
        game_id, round_id, player_roles, protagonist, public_history,
        dialogue_gemini3, tactic_scale_gemini3,
        dialogue_gpt52, tactic_scale_gpt52
    )
    
    if result is None:
        print(f"  ✗ Error: {raw_response}")
        verification_stats['errors'] += 1
        retry_rows.append({'idx': idx, 'game_id': game_id, 'round_id': round_id, 'reason': 'api_error'})
        continue
    
    verification_stats['total_games'] += 1
    
    # Determine LLM source and label for output (map Response A/B back to actual model names)
    if result['recommendation'] == 'RESPONSE_A':
        llm_used = 'Gemini-3.1'
        selected_row = gemini3_df.iloc[idx].copy()
        verification_stats['response_1_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_A':
        llm_used = 'Gemini-3.1-Claude-4.5'
        selected_row = gemini3_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        verification_stats['targeted_correction_1'] += 1
    elif result['recommendation'] == 'RESPONSE_B':
        llm_used = 'GPT-5.2'
        selected_row = gpt52_df.iloc[idx].copy()
        verification_stats['response_2_selected'] += 1
    elif result['recommendation'] == 'CORRECTION_B':
        llm_used = 'GPT-5.2-Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        verification_stats['targeted_correction_2'] += 1
    else:  # CUSTOM_GENERATION
        llm_used = 'Claude-4.5'
        selected_row = gpt52_df.iloc[idx].copy()
        selected_row['discussion_log'] = result['selected_dialogue']
        verification_stats['full_custom_generation'] += 1
    
    # Use the already-ordered tactic scale from verify_dialogue_pair (fallback to GPT-5.2)
    selected_tactic_scale = result['selected_tactic_scale'] or format_matrix_tactic_scale(tactic_scale_gpt52)
    
    # Track pairwise tiebreakers separately
    if result['correction_mode'] == 'Pairwise_Tiebreaker':
        verification_stats['pairwise_tiebreaker'] += 1
    
    # Update matrix tactic scale in selected row
    selected_row['matrix_tactic_scale'] = selected_tactic_scale
    
    # Add LLM_used and Claude reasoning columns
    selected_row['LLM_used'] = llm_used
    selected_row['Claude_Reasoning'] = result['reasoning']
    verified_results.append(selected_row)
    
    # Queue for retry if verify_dialogue_pair detected a silent failure
    if result.get('needs_retry'):
        retry_rows.append({'idx': idx, 'game_id': game_id, 'round_id': round_id, 'reason': result['correction_mode']})
        print(f"  ⚠️  Queued for Step 4b retry (reason: {result['correction_mode']})")
    
    # Create detailed criteria record for CSV
    criteria_record = {
        'ID': score_id,
        'Gemini31_Coherence': format_check(result['gemini3_criteria']['coherence']),
        'Gemini31_History': format_check(result['gemini3_criteria']['history']),
        'Gemini31_Matrix': format_check(result['gemini3_criteria']['matrix']),
        'Gemini31_Authenticity': format_check(result['gemini3_criteria']['authenticity']),
        'Gemini31_Format': format_check(result['gemini3_criteria']['format']),
        'Gemini31_Total': f"{result['gemini3_total']} of 5",
        'GPT52_Coherence': format_check(result['gpt52_criteria']['coherence']),
        'GPT52_History': format_check(result['gpt52_criteria']['history']),
        'GPT52_Matrix': format_check(result['gpt52_criteria']['matrix']),
        'GPT52_Authenticity': format_check(result['gpt52_criteria']['authenticity']),
        'GPT52_Format': format_check(result['gpt52_criteria']['format']),
        'GPT52_Total': f"{result['gpt52_total']} of 5",
        'Selected_LLM': llm_used,
        'Correction_Mode': result['correction_mode'] or 'None',
        'Claude_Reasoning': result['reasoning'],
        'Gemini31_Criteria_Explanations': json.dumps(result['gemini3_criteria_explanations']),
        'GPT52_Criteria_Explanations': json.dumps(result['gpt52_criteria_explanations']),
        'Claude_Response': raw_response  # Store full raw response for debugging
    }
    criteria_results.append(criteria_record)
    
    # Print summary
    print(f"  ✓ Gemini-3.1: {result['gemini3_total']}/5 pass ({', '.join([k for k,v in result['gemini3_criteria'].items() if v])})")
    print(f"  ✓ GPT-5.2: {result['gpt52_total']}/5 pass ({', '.join([k for k,v in result['gpt52_criteria'].items() if v])})")
    print(f"  ✓ Selected: {llm_used}")
    if result['correction_mode']:
        print(f"  ✓ Correction Mode: {result['correction_mode']}")
    print(f"  ✓ Reason: {result['reasoning'][:100]}...")
    
    # Rate limit to avoid API throttling
    time.sleep(1)

print("\n" + "=" * 80)
print(f"Verification Summary (first {TEST_LIMIT} games):")
print(f"  Total verified: {verification_stats['total_games']}")
print(f"  Gemini-3.1 selected (original): {verification_stats['response_1_selected']}")
print(f"  GPT-5.2 selected (original): {verification_stats['response_2_selected']}")
print(f"  Pairwise tiebreaker used: {verification_stats['pairwise_tiebreaker']}")
print(f"  Gemini-3.1 + Targeted correction: {verification_stats['targeted_correction_1']}")
print(f"  GPT-5.2 + Targeted correction: {verification_stats['targeted_correction_2']}")
print(f"  Claude 4.5 full custom generation: {verification_stats['full_custom_generation']}")
print(f"  Errors: {verification_stats['errors']}")
print(f"  ⚠️  Queued for Step 4b retry: {len(retry_rows)}")


Starting verification of 225 games...

[1/225] Verifying game G026/1...
  ✓ Gemini-3.1: 2/5 pass (history, format)
  ✓ GPT-5.2: 5/5 pass (coherence, history, matrix, authenticity, format)
  ✓ Selected: GPT-5.2
  ✓ Reason: Response B passes all 5 criteria (5/5) while Response A fails coherence, matrix alignment, and authe...

[2/225] Verifying game G027/1...
  ✓ Gemini-3.1: 2/5 pass (history, format)
  ✓ GPT-5.2: 5/5 pass (coherence, history, matrix, authenticity, format)
  ✓ Selected: GPT-5.2
  ✓ Reason: Response B passes all 5 criteria (5/5) while Response A fails on coherence, matrix alignment, and au...

[3/225] Verifying game G028/1...
  ✓ Gemini-3.1: 4/5 pass (history, matrix, authenticity, format)
  ✓ GPT-5.2: 4/5 pass (coherence, matrix, authenticity, format)
  ✓ Selected: Gemini-3.1-Claude-4.5
  ✓ Correction Mode: Targeted
  ✓ Reason: Both responses score 4/5. Response A fails coherence (P3's premature nuclear lie is tactically impla...

[4/225] Verifying game G029/1...
  ✓ Gem


## Step 4b: Retry Failed Rows

Retries rows collected in `retry_rows` during Step 4a. Three failure reasons are tracked:

| Reason | Cause | Entry in `verified_results`? |
|---|---|---|
| `api_error` | `verify_dialogue_pair` returned `None` (API / parse failure) | ❌ No — must append if retry succeeds |
| `Targeted` | `tc_dialogue` was `None`; targeted correction silently fell back to uncorrected base dialogue | ✅ Yes — patch in-place |
| `Full_Custom` | `cd_dialogue` was empty; `selected_dialogue` stored as `None` | ✅ Yes — patch in-place |

Rows that still fail after retry receive a hard fallback (raw model dialogue by higher score), logged with a `-Fallback` suffix in `LLM_used`.


In [ ]:

# ─── Step 4b: Retry failed rows ──────────────────────────────────────────────
#
# Uses retry_rows list built during Step 4a (cell 12).
# Failure types:
#   'api_error'   — no entry in verified_results; append if retry succeeds
#   'Targeted'    — entry exists but dialogue is uncorrected fallback; patch
#   'Full_Custom' — entry exists but discussion_log is None; patch
# ─────────────────────────────────────────────────────────────────────────────

_CRITERIA_ORDER = ['coherence', 'history', 'matrix', 'authenticity', 'format']
_LABEL_MAP = {
    'RESPONSE_A':        'Gemini-3.1',
    'CORRECTION_A':      'Gemini-3.1-Claude-4.5',
    'RESPONSE_B':        'GPT-5.2',
    'CORRECTION_B':      'GPT-5.2-Claude-4.5',
    'CUSTOM_GENERATION': 'Claude-4.5',
}

def _make_criteria_record(score_id, result, llm_used, correction_mode, raw_response):
    """Build a criteria_results record from a verify_dialogue_pair result dict."""
    return {
        'ID':                             score_id,
        'Gemini31_Coherence':             format_check(result['gemini3_criteria']['coherence']),
        'Gemini31_History':               format_check(result['gemini3_criteria']['history']),
        'Gemini31_Matrix':                format_check(result['gemini3_criteria']['matrix']),
        'Gemini31_Authenticity':          format_check(result['gemini3_criteria']['authenticity']),
        'Gemini31_Format':                format_check(result['gemini3_criteria']['format']),
        'Gemini31_Total':                 f"{result['gemini3_total']} of 5",
        'GPT52_Coherence':                format_check(result['gpt52_criteria']['coherence']),
        'GPT52_History':                  format_check(result['gpt52_criteria']['history']),
        'GPT52_Matrix':                   format_check(result['gpt52_criteria']['matrix']),
        'GPT52_Authenticity':             format_check(result['gpt52_criteria']['authenticity']),
        'GPT52_Format':                   format_check(result['gpt52_criteria']['format']),
        'GPT52_Total':                    f"{result['gpt52_total']} of 5",
        'Selected_LLM':                   llm_used,
        'Correction_Mode':                correction_mode,
        'Claude_Reasoning':               result['reasoning'],
        'Gemini31_Criteria_Explanations': json.dumps(result['gemini3_criteria_explanations']),
        'GPT52_Criteria_Explanations':    json.dumps(result['gpt52_criteria_explanations']),
        'Claude_Response':                raw_response,
    }

def _make_error_criteria_stub(score_id, raw_response):
    """Build an all-zero criteria stub for a row where the API failed entirely."""
    zero_expl = json.dumps({c: '' for c in _CRITERIA_ORDER})
    return {
        'ID':                             score_id,
        'Gemini31_Coherence':             0,
        'Gemini31_History':               0,
        'Gemini31_Matrix':                0,
        'Gemini31_Authenticity':          0,
        'Gemini31_Format':                0,
        'Gemini31_Total':                 '0 of 5',
        'GPT52_Coherence':                0,
        'GPT52_History':                  0,
        'GPT52_Matrix':                   0,
        'GPT52_Authenticity':             0,
        'GPT52_Format':                   0,
        'GPT52_Total':                    '0 of 5',
        'Selected_LLM':                   'Error',
        'Correction_Mode':                'Error',
        'Claude_Reasoning':               f'(api_error: {raw_response})',
        'Gemini31_Criteria_Explanations': zero_expl,
        'GPT52_Criteria_Explanations':    zero_expl,
        'Claude_Response':                '',
    }

if not retry_rows:
    print("✅ No rows to retry — all verifications succeeded.")
else:
    n_api    = sum(1 for r in retry_rows if r['reason'] == 'api_error')
    n_tc     = sum(1 for r in retry_rows if r['reason'] == 'Targeted')
    n_cd     = sum(1 for r in retry_rows if r['reason'] == 'Full_Custom')
    print(f"⚠️  Retrying {len(retry_rows)} row(s): "
          f"{n_api} api_error, {n_tc} Targeted, {n_cd} Full_Custom\n")
    for entry in retry_rows:
        print(f"  • {entry['game_id']}/{entry['round_id']} — {entry['reason']}")
    print()

    succeeded    = 0
    still_failed = 0

    for entry in sorted(retry_rows, key=lambda e: e['idx']):
        idx      = entry['idx']
        game_id  = entry['game_id']
        round_id = entry['round_id']
        reason   = entry['reason']
        score_id = f"{game_id}/{round_id}"

        print(f" Retrying {score_id} (reason: {reason})...")

        player_roles       = gemini3_df.iloc[idx]['player_roles']
        protagonist        = gemini3_df.iloc[idx]['role_id']
        public_history     = gemini3_df.iloc[idx]['public_history']
        dialogue_gemini3   = gemini3_df.iloc[idx]['discussion_log']
        tactic_scale_gem3  = gemini3_df.iloc[idx]['matrix_tactic_scale']
        dialogue_gpt52     = gpt52_df.iloc[idx]['discussion_log']
        tactic_scale_gpt52 = gpt52_df.iloc[idx]['matrix_tactic_scale']

        result, raw_response = verify_dialogue_pair(
            game_id, round_id,
            player_roles, protagonist, public_history,
            dialogue_gemini3, tactic_scale_gem3,
            dialogue_gpt52,   tactic_scale_gpt52
        )

        # Find existing position in verified_results / criteria_results
        existing_pos = next(
            (i for i, r in enumerate(verified_results) if r.get('game_id') == game_id),
            None
        )
        existing_cr_pos = next(
            (i for i, cr in enumerate(criteria_results) if cr['ID'] == score_id),
            None
        )

        # ── API / parse error on retry ─────────────────────────────────────
        if result is None:
            print(f"  ✗ Retry failed: {raw_response}")
            if existing_pos is not None:
                print(f"  ↩  Keeping previous fallback entry in verified_results (pos {existing_pos}).")
            else:
                print(f"  ✗  No entry created for {game_id} — game will be absent from CSV.")

            # Keep criteria_results consistent: write/overwrite an error stub
            stub = _make_error_criteria_stub(score_id, raw_response)
            if existing_cr_pos is not None:
                criteria_results[existing_cr_pos] = stub
            else:
                criteria_results.append(stub)
            print(f"  ↩  Criteria stub (all-zero) written for {score_id}.")

            still_failed += 1
            print()
            time.sleep(1)
            continue

        # ── Still a silent failure after retry → hard fallback ─────────────
        if result.get('needs_retry') or not result.get('selected_dialogue'):
            g3_total = result['gemini3_total']
            gp_total = result['gpt52_total']
            if g3_total >= gp_total:
                fallback_dialogue = dialogue_gemini3
                fallback_ts       = tactic_scale_gem3
                fallback_llm      = 'Gemini-3.1-Fallback'
                fallback_base     = gemini3_df.iloc[idx].copy()
            else:
                fallback_dialogue = dialogue_gpt52
                fallback_ts       = tactic_scale_gpt52
                fallback_llm      = 'GPT-5.2-Fallback'
                fallback_base     = gpt52_df.iloc[idx].copy()
            fallback_base['discussion_log']      = fallback_dialogue
            fallback_base['matrix_tactic_scale'] = fallback_ts
            fallback_base['LLM_used']             = fallback_llm
            fallback_base['Claude_Reasoning']     = '(fallback: still empty after retry)'

            if existing_pos is not None:
                verified_results[existing_pos] = fallback_base
                print(f"  ↩  Patched verified_results [pos {existing_pos}] with fallback: {fallback_llm}")
            else:
                verified_results.append(fallback_base)
                print(f"  ↩  Appended fallback entry: {fallback_llm}")

            # Write actual scores to criteria_results (we have them even if dialogue failed)
            cr = _make_criteria_record(score_id, result, fallback_llm, 'Fallback', raw_response)
            if existing_cr_pos is not None:
                criteria_results[existing_cr_pos] = cr
            else:
                criteria_results.append(cr)
            print(f"  ↩  Criteria record (real scores) written with Correction_Mode=Fallback.")

            still_failed += 1
            print()
            time.sleep(1)
            continue

        # ── Successful retry: build selected_row ──────────────────────────
        rec      = result['recommendation']
        llm_used = _LABEL_MAP.get(rec, rec)

        if rec == 'RESPONSE_A':
            selected_row = gemini3_df.iloc[idx].copy()
        elif rec == 'CORRECTION_A':
            selected_row = gemini3_df.iloc[idx].copy()
            selected_row['discussion_log'] = result['selected_dialogue']
        elif rec == 'RESPONSE_B':
            selected_row = gpt52_df.iloc[idx].copy()
        elif rec == 'CORRECTION_B':
            selected_row = gpt52_df.iloc[idx].copy()
            selected_row['discussion_log'] = result['selected_dialogue']
        else:  # CUSTOM_GENERATION
            selected_row = gpt52_df.iloc[idx].copy()
            selected_row['discussion_log'] = result['selected_dialogue']

        selected_row['matrix_tactic_scale'] = (
            result['selected_tactic_scale'] or format_matrix_tactic_scale(tactic_scale_gpt52)
        )
        selected_row['LLM_used']         = llm_used
        selected_row['Claude_Reasoning'] = result['reasoning']

        g3_pass = ', '.join(k for k in _CRITERIA_ORDER if result['gemini3_criteria'].get(k))
        gp_pass = ', '.join(k for k in _CRITERIA_ORDER if result['gpt52_criteria'].get(k))
        print(f"  ✓ Gemini-3.1: {result['gemini3_total']}/5 ({g3_pass})")
        print(f"  ✓ GPT-5.2:    {result['gpt52_total']}/5 ({gp_pass})")
        print(f"  ✓ Selected: {llm_used} | Mode: {result['correction_mode'] or 'None'}")

        # Patch or append verified_results
        if existing_pos is not None:
            old_llm = verified_results[existing_pos].get('LLM_used', '?')
            verified_results[existing_pos] = selected_row
            print(f"  ✓ Patched verified_results [pos {existing_pos}] → {llm_used} (was: {old_llm})")
        else:
            verified_results.insert(idx, selected_row)
            print(f"  ✓ Inserted new entry at pos {idx} → {llm_used}")

        # Patch or insert criteria_results
        new_cr = _make_criteria_record(
            score_id, result, llm_used, result['correction_mode'] or 'None', raw_response
        )
        if existing_cr_pos is not None:
            criteria_results[existing_cr_pos] = new_cr
        else:
            criteria_results.insert(idx, new_cr)

        succeeded += 1
        print()
        time.sleep(1)

    print("─" * 60)
    print(f"✅ Retry complete: {succeeded} succeeded, {still_failed} still failed.")
    if still_failed:
        print("   Persistent failures use raw-model fallback dialogue (hard fallback) or zero-stub (api_error).")
    print(f"   verified_results: {len(verified_results)} rows | criteria_results: {len(criteria_results)} rows")
    print("   verified_results and criteria_results are ready to save.")


⚠️  Retrying 2 row(s): 1 api_error, 0 Targeted, 1 Full_Custom

  • G073/1 — api_error
  • G188/1 — Full_Custom

 Retrying G073/1 (reason: api_error)...
  ✓ Gemini-3.1: 3/5 (history, authenticity, format)
  ✓ GPT-5.2:    5/5 (coherence, history, matrix, authenticity, format)
  ✓ Selected: GPT-5.2 | Mode: None
  ✓ Appended new entry → GPT-5.2

 Retrying G188/1 (reason: Full_Custom)...
  ✓ Gemini-3.1: 3/5 (coherence, authenticity, format)
  ✓ GPT-5.2:    4/5 (coherence, matrix, authenticity, format)
  ✓ Selected: GPT-5.2-Claude-4.5 | Mode: Targeted
  ✓ Patched verified_results [pos 161] → GPT-5.2-Claude-4.5 (was: Claude-4.5)

────────────────────────────────────────────────────────────
✅ Retry complete: 2 succeeded, 0 still failed.
   verified_results: 225 rows | criteria_results: 225 rows
   verified_results and criteria_results are ready to save.


## Step 4c: Re-check Corrected and Custom-Generated Rows

Rows with `Correction_Mode` in `{Targeted, Full_Custom}` went through a **one-way** correction process during Step 4a/4b — Claude fixed or generated them, but the result was never independently scored. This step closes that loop.

| Correction Mode | Approx. Count | Risk |
|---|---|---|
| `Targeted` | ~41 | Fixing one criterion may silently break another |
| `Full_Custom` | ~2 | Custom generation was never scored at all |

**Single-response evaluation via `verify_single_dialogue()`:**

| Recheck Score | Action | `Recheck_4c` tag |
|---|---|---|
| 5/5 | Accept as-is — no change to `verified_results` | `ACCEPTED` |
| 4/5 | One targeted correction of the failing criterion | `RECORRECTED` |
| ≤3/5 | Full regeneration by Claude | `REGENERATED` |

- `LLM_used` is updated with a `-4cFixed` or `-4cRegen` suffix for changed rows
- `Recheck_4c`, `Recheck_4c_Score`, `Recheck_4c_Reasoning`, `Recheck_4c_Explanations` columns added to `criteria_results`
- All patches happen in-place; CSV save in Step 5 picks them up automatically
- This cell is **idempotent at scale**: only processes Targeted/Full_Custom rows, regardless of total batch size

In [ ]:
# ─── Step 4c: verify_single_dialogue() ───────────────────────────────────────
#
# Evaluates ONE dialogue on the same 5 binary criteria as verify_dialogue_pair().
# Called only for rows with Correction_Mode in {Targeted, Full_Custom}.
#
# Returns: (result_dict, raw_response)
# result_dict keys:
#   criteria              — {coherence, history, matrix, authenticity, format} → bool
#   criteria_explanations — {coherence, ...} → str
#   total                 — int 0–5
#   recommendation        — 'ACCEPT' | 'TARGETED_CORRECTION' | 'FULL_REGENERATION'
#   failed_criterion      — str or None
#   reasoning             — str
#   selected_dialogue     — str (unchanged / corrected / regenerated)
#   selected_tactic_scale — str
#   correction_mode       — 'Accepted' | 'Recorrected' | 'Regenerated'
#   needs_retry           — bool
# ─────────────────────────────────────────────────────────────────────────────

def verify_single_dialogue(game_id, round_id, player_roles, protagonist,
                            public_history, discussion_log, matrix_tactic_scale):
    recheck_prompt = f"""You are evaluating a single AI-generated Avalon discussion log for quality. The dialogue may have already undergone one round of correction. Score it on 5 binary criteria and, if it fails any, apply the minimum fix required.

**CONTEXT:**
Game: {game_id}, Round {round_id}
Player Roles: {player_roles}
Protagonist (does NOT speak in dialogue): {protagonist}

Public History (up to current round):
{public_history}

---

**DIALOGUE TO EVALUATE:**

Discussion Log:
{discussion_log}

Matrix Tactic Scale:
{matrix_tactic_scale}

---

**TACTIC ANNOTATION FRAMEWORK:**
{TACTIC_TAXONOMY_REF}

Each speaker's annotation in matrix_tactic_scale must use this structure:
  "PX": {{"row": "<Information Strategy>", "col": "<Goal Orientation>", "tactic": "<exact name from taxonomy>", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}

---

**EVALUATION CRITERIA (1 = PASS, 0 = FAIL):**

1. **Discussion Coherence & Tactic Situational Fit**:
   1 = Each player's tactic makes situational sense given the social dynamics of the discussion — their conversational position, what other players are saying, and how tensions develop across the exchange
   0 = A player's tactic is socially incongruous given the discussion flow (e.g., "Pragmatic silence" for a player being directly accused by multiple others; "Vague filler" for a player everyone is turning to for information). The utterance may match the label literally, but the tactic choice is implausible for that social moment.

2. **Public History Alignment**:
   1 = Dialogue contextually fits quest outcomes, team compositions, and voting patterns
   0 = Ignores prior events or contradicts established game state

3. **Tactic-Dialogue Alignment**:
   1 = Each player's annotation accurately describes what they actually said: the tactic name is a valid entry from the taxonomy, the row/col match the tactic's position in the 4×4 matrix, scale is GRS for Good players and Mach-IV for Evil players, and the level (High/Moderate/Low) is plausible given the sophistication of the utterance
   0 = The annotation mislabels what the player actually said (e.g., "Pragmatic silence" for a player who gave a 4-sentence analysis; "Hard lying" for a player who spoke honestly), OR a tactic name is not in the valid taxonomy, OR scale/row/col are inconsistent with the tactic definition
   Framing: the dialogue is the ground truth — the annotation must describe what was written.

4. **Authenticity & Skill Consistency**:
   1 = The dialogue reads as a believable Avalon exchange with natural accusation/defense patterns, and the linguistic sophistication of each utterance is consistent with the speaker's assigned skill level.
   0 = The dialogue feels stilted or artificial, or the sophistication of an utterance clearly mismatches the assigned level (e.g., elaborate strategic reasoning from a Low-skill player, or uncertain hedging from a High-skill player).

5. **Dialogue Format Compliance**:
   1 = Starts with "Discussion after Quest X:", exactly 4 speakers (all non-protagonists), each speaks exactly once, in this format:
      Discussion after Quest X:
      PlayerA: [dialogue]
      PlayerB: [dialogue]
      PlayerC: [dialogue]
      PlayerD: [dialogue]
   0 = Wrong or missing header, incorrect speaker count, protagonist appears as speaker, or any format violations

---

**DECISION LOGIC:**

1. **5/5** → ACCEPT the dialogue as-is
2. **4/5** → TARGETED_CORRECTION: fix the single failing criterion in the provided dialogue
3. **≤3/5** → FULL_REGENERATION: generate a completely new dialogue that passes all 5 criteria

---

**REQUIRED OUTPUT FORMAT:**

Return a valid JSON object:

```json
{{
  "criteria": {{
    "coherence": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "history": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "matrix": {{"score": 0, "explanation": "Brief explanation why it passed or failed"}},
    "authenticity": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}},
    "format": {{"score": 1, "explanation": "Brief explanation why it passed or failed"}}
  }},
  "total": 4,
  "recommendation": "TARGETED_CORRECTION",
  "reasoning": "2-3 sentences on overall quality and decision",
  "failed_criterion": "Matrix",
  "targeted_correction": {{
    "dialogue": "Full corrected dialogue text fixing only the failed criterion",
    "matrix_tactic_scale": {{"PX": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }},
  "regenerated_dialogue": null
}}
```

When recommendation is FULL_REGENERATION, regenerated_dialogue takes this structure (targeted_correction will be null):

```json
{{
  ...,
  "recommendation": "FULL_REGENERATION",
  "targeted_correction": null,
  "regenerated_dialogue": {{
    "dialogue": "Discussion after Quest X:\\nPlayerA: [utterance]\\nPlayerB: [utterance]\\nPlayerC: [utterance]\\nPlayerD: [utterance]",
    "matrix_tactic_scale": {{"PX": {{"row": "...", "col": "...", "tactic": "...", "scale": "GRS|Mach-IV", "level": "High|Moderate|Low"}}, "...": {{...}}}}
  }}
}}
```

**Field rules:**
- **targeted_correction**: Populated only when recommendation is TARGETED_CORRECTION; an object with:
  - "dialogue": full corrected dialogue text fixing only the failed criterion
  - "matrix_tactic_scale": JSON mapping each speaker to their tactic annotation; keep all speakers' entries from the input unless the correction strictly requires a change
  Otherwise null.
- **regenerated_dialogue**: Populated only when recommendation is FULL_REGENERATION; an object with:
  - "dialogue": complete new dialogue (exactly 4 speakers, protagonist excluded, "Discussion after Quest X:" format)
  - "matrix_tactic_scale": JSON mapping each speaker to their tactic annotation; use the input matrix_tactic_scale as baseline — keep each speaker's row, col, tactic, scale, and level unless the new dialogue strictly requires a change
  Otherwise null.
- **failed_criterion**: The single failing criterion name (Coherence/History/Matrix/Authenticity/Format) when recommendation is TARGETED_CORRECTION; null otherwise
- Return ONLY the JSON object, no markdown code blocks, no extra text."""

    try:
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": recheck_prompt}]
        )
        response_text = response.content[0].text

        try:
            clean_text = response_text.strip()
            if clean_text.startswith('```'):
                lines = clean_text.split('\n')
                lines = lines[1:]
                for i, line in enumerate(lines):
                    if line.strip() == '```':
                        lines = lines[:i]
                        break
                clean_text = '\n'.join(lines)

            data     = json.loads(clean_text)
            criteria = data['criteria']

            result = {
                'criteria': {k: criteria[k]['score'] == 1 for k in _CRITERIA_ORDER},
                'criteria_explanations': {k: criteria[k].get('explanation', '') for k in _CRITERIA_ORDER},
                'total': int(data['total']) if isinstance(data['total'], int) else int(str(data['total']).split('/')[0]),
                'recommendation':    data.get('recommendation', 'ACCEPT'),
                'reasoning':         data.get('reasoning', ''),
                'failed_criterion':  data.get('failed_criterion'),
                'targeted_correction': data.get('targeted_correction'),
                'regenerated_dialogue': data.get('regenerated_dialogue'),
                'selected_dialogue':     None,
                'selected_tactic_scale': None,
                'correction_mode':       None,
                'needs_retry':           False,
            }

            rec = result['recommendation']

            if result['total'] == 5 or rec == 'ACCEPT':
                result['correction_mode']       = 'Accepted'
                result['selected_dialogue']     = discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(discussion_log, matrix_tactic_scale)

            elif rec == 'TARGETED_CORRECTION':
                tc          = result.get('targeted_correction') or {}
                tc_dialogue = tc.get('dialogue') if isinstance(tc, dict) else None
                tc_matrix   = tc.get('matrix_tactic_scale') if isinstance(tc, dict) else None
                if tc_dialogue is None:
                    print(f"  ⚠️  WARNING: targeted_correction.dialogue is None for {game_id} — flagging for retry")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Recorrected'
                result['selected_dialogue']     = tc_dialogue or discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    tc_dialogue or discussion_log,
                    json.dumps(tc_matrix, separators=(',', ':')) if isinstance(tc_matrix, dict) else (tc_matrix or matrix_tactic_scale)
                )

            else:  # FULL_REGENERATION
                rd          = result.get('regenerated_dialogue') or {}
                rd_dialogue = rd.get('dialogue') if isinstance(rd, dict) else None
                rd_matrix   = rd.get('matrix_tactic_scale') if isinstance(rd, dict) else None
                if not rd_dialogue:
                    print(f"  ⚠️  WARNING: regenerated_dialogue is empty for {game_id}!")
                    result['needs_retry'] = True
                result['correction_mode']       = 'Regenerated'
                result['selected_dialogue']     = rd_dialogue or discussion_log
                result['selected_tactic_scale'] = reorder_matrix_by_speaking_order(
                    rd_dialogue or discussion_log,
                    json.dumps(rd_matrix, separators=(',', ':')) if isinstance(rd_matrix, dict) else (rd_matrix or matrix_tactic_scale)
                )

            return result, response_text

        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed for {game_id}: {e}")
            return None, f"JSON parsing error: {e}"
        except KeyError as e:
            print(f"❌ Missing JSON field for {game_id}: {e}")
            return None, f"Missing field: {e}"
        except Exception as e:
            print(f"❌ Error processing recheck JSON for {game_id}: {e}")
            return None, str(e)

    except Exception as e:
        print(f"  Error rechecking game {game_id}: {str(e)}")
        return None, str(e)

print("✓ verify_single_dialogue() defined")

# ─── Step 4c: Main recheck loop ───────────────────────────────────────────────
#
# Rechecks rows with Correction_Mode in {Targeted, Full_Custom} using the
# verify_single_dialogue() function defined above.
# Patches verified_results and criteria_results in-place.
# No bootstrap needed — criteria_results / verified_results are live from Steps 4a/4b.
# ─────────────────────────────────────────────────────────────────────────────

recheck_rows = [
    cr for cr in criteria_results
    if cr.get('Correction_Mode') in ('Targeted', 'Full_Custom')
]

if not recheck_rows:
    print("✅ No corrected rows to recheck — all verifications were direct selections.")
else:
    n_targeted = sum(1 for cr in recheck_rows if cr['Correction_Mode'] == 'Targeted')
    n_custom   = sum(1 for cr in recheck_rows if cr['Correction_Mode'] == 'Full_Custom')
    print(f"🔍 Rechecking {len(recheck_rows)} corrected/custom rows: "
          f"{n_targeted} Targeted, {n_custom} Full_Custom\n")

    accepted    = 0
    recorrected = 0
    regenerated = 0
    still_failed = 0

    for cr in recheck_rows:
        score_id              = cr['ID']
        game_id, round_id_str = score_id.split('/')
        round_id              = int(round_id_str)

        # Pull context from original DataFrames (gemini3_df is source of truth for game context)
        row_mask = gemini3_df['game_id'] == game_id
        if not row_mask.any():
            print(f"  ⚠️  {score_id}: game not found in gemini3_df — skipping")
            continue
        idx            = gemini3_df.index[row_mask][0]
        player_roles   = gemini3_df.loc[idx, 'player_roles']
        protagonist    = gemini3_df.loc[idx, 'role_id']
        public_history = gemini3_df.loc[idx, 'public_history']

        # Pull current (already-corrected) dialogue from verified_results
        vr_pos = next(
            (i for i, r in enumerate(verified_results) if r.get('game_id') == game_id),
            None
        )
        if vr_pos is None:
            print(f"  ⚠️  {score_id}: not found in verified_results — skipping")
            continue
        current_dialogue = verified_results[vr_pos]['discussion_log']
        current_matrix   = verified_results[vr_pos]['matrix_tactic_scale']

        print(f"  Rechecking {score_id} "
              f"(was: {cr['Correction_Mode']}, LLM: {cr['Selected_LLM']})...")

        result, raw_response = verify_single_dialogue(
            game_id, round_id, player_roles, protagonist, public_history,
            current_dialogue, current_matrix
        )

        # ── API error on recheck ───────────────────────────────────────────
        if result is None:
            print(f"  ✗ Recheck API error for {score_id}: {raw_response}")
            cr['Recheck_4c']              = 'API_ERROR'
            cr['Recheck_4c_Score']        = '0 of 5'
            cr['Recheck_4c_Reasoning']    = f'(api_error: {raw_response})'
            cr['Recheck_4c_Explanations'] = '{}'
            still_failed += 1
            print()
            time.sleep(1)
            continue

        recheck_score = result['total']
        mode          = result['correction_mode']   # 'Accepted' | 'Recorrected' | 'Regenerated'

        # Write recheck metadata to criteria_results entry (in-place)
        cr['Recheck_4c']              = mode.upper()
        cr['Recheck_4c_Score']        = f"{recheck_score} of 5"
        cr['Recheck_4c_Reasoning']    = result['reasoning']
        cr['Recheck_4c_Explanations'] = json.dumps(result['criteria_explanations'])

        pass_str = ', '.join(k for k in _CRITERIA_ORDER if result['criteria'].get(k))
        print(f"  ✓ Recheck score: {recheck_score}/5 ({pass_str}) → {mode}")

        if mode == 'Accepted':
            accepted += 1
            # No change to verified_results; dialogue was already correct

        else:
            old_llm    = verified_results[vr_pos].get('LLM_used', '?')
            llm_suffix = '-4cFixed' if mode == 'Recorrected' else '-4cRegen'
            new_llm    = old_llm + llm_suffix

            verified_results[vr_pos]['discussion_log']      = result['selected_dialogue']
            verified_results[vr_pos]['matrix_tactic_scale'] = result['selected_tactic_scale']
            verified_results[vr_pos]['LLM_used']            = new_llm
            verified_results[vr_pos]['Claude_Reasoning']    = (
                str(verified_results[vr_pos].get('Claude_Reasoning', ''))
                + f" | 4c: {result['reasoning']}"
            )
            cr['Selected_LLM'] = new_llm

            print(f"  ✓ Patched verified_results [pos {vr_pos}]: {old_llm} → {new_llm}")
            if mode == 'Recorrected':
                recorrected += 1
            else:
                regenerated += 1

        print()
        time.sleep(1)

    print("─" * 60)
    print(f"✅ Step 4c complete:")
    print(f"   Accepted  (5/5, no change):   {accepted}")
    print(f"   Recorrected (4/5 targeted):   {recorrected}")
    print(f"   Regenerated (≤3/5 regen):     {regenerated}")
    print(f"   Still failed (API error):     {still_failed}")
    print(f"   verified_results: {len(verified_results)} rows | criteria_results: {len(criteria_results)} rows")
    print("   Ready for Step 5 (Create Output DataFrames).")

✓ verify_single_dialogue() defined — used by log-gen-verifier-4c.ipynb (R1 recheck) and future R2-R5 verifier notebooks.


## Step 5: Create Output DataFrames

Convert verification results into structured DataFrames.

In [9]:
# Create Output DataFrames
# Create DataFrame from verified results (full data)
if verified_results:
    verified_df = pd.DataFrame(verified_results)
    
    # Ensure LLM_used is last column
    cols = verified_df.columns.tolist()
    cols = [col for col in cols if col != 'LLM_used'] + ['LLM_used']
    verified_df = verified_df[cols]
    
    print(f"\n✓ Verified DataFrame (combined) shape: {verified_df.shape}")
    print(f"✓ Columns: {list(verified_df.columns)}")
    print(f"\nFirst verified row (LLM_used column):")
    print(verified_df[['game_id', 'round_id', 'LLM_used']].head())
else:
    print("No results to create verified output CSV")
    verified_df = None

# Create DataFrame from criteria results (detailed per-criterion tracking)
if criteria_results:
    criteria_df = pd.DataFrame(criteria_results)
    
    print(f"\n✓ Criteria DataFrame shape: {criteria_df.shape}")
    print(f"✓ Columns: {list(criteria_df.columns)}")
    print(f"\nFirst criteria records:")
    print(criteria_df.head())
    
    # Calculate aggregate statistics
    print(f"\n{'='*60}")
    print("AGGREGATE STATISTICS (for paper analysis):")
    print(f"{'='*60}")
    
    # Helper function to count passes
    def count_passes(df, prefix):
        coherence = (df[f'{prefix}_Coherence'] == 1).sum()
        history = (df[f'{prefix}_History'] == 1).sum()
        matrix = (df[f'{prefix}_Matrix'] == 1).sum()
        authenticity = (df[f'{prefix}_Authenticity'] == 1).sum()
        format_check = (df[f'{prefix}_Format'] == 1).sum()
        total = len(df)
        return {
            'Coherence': f"{coherence}/{total} ({coherence/total*100:.1f}%)",
            'History': f"{history}/{total} ({history/total*100:.1f}%)",
            'Matrix': f"{matrix}/{total} ({matrix/total*100:.1f}%)",
            'Authenticity': f"{authenticity}/{total} ({authenticity/total*100:.1f}%)",
            'Format': f"{format_check}/{total} ({format_check/total*100:.1f}%)"
        }
    
    gemini31_stats = count_passes(criteria_df, 'Gemini31')
    gpt52_stats = count_passes(criteria_df, 'GPT52')
    
    print("\nGemini-3.1 Pass Rates:")
    for criterion, stat in gemini31_stats.items():
        print(f"  {criterion:15s}: {stat}")
    
    print("\nGPT-5.2 Pass Rates:")
    for criterion, stat in gpt52_stats.items():
        print(f"  {criterion:15s}: {stat}")
else:
    print("No results to create criteria output CSV")
    criteria_df = None


✓ Verified DataFrame (combined) shape: (225, 11)
✓ Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'Claude_Reasoning', 'LLM_used']

First verified row (LLM_used column):
  game_id  round_id               LLM_used
0    G026         1                GPT-5.2
1    G027         1                GPT-5.2
2    G028         1  Gemini-3.1-Claude-4.5
3    G029         1                GPT-5.2
4    G030         1                GPT-5.2

✓ Criteria DataFrame shape: (225, 19)
✓ Columns: ['ID', 'Gemini31_Coherence', 'Gemini31_History', 'Gemini31_Matrix', 'Gemini31_Authenticity', 'Gemini31_Format', 'Gemini31_Total', 'GPT52_Coherence', 'GPT52_History', 'GPT52_Matrix', 'GPT52_Authenticity', 'GPT52_Format', 'GPT52_Total', 'Selected_LLM', 'Correction_Mode', 'Claude_Reasoning', 'Gemini31_Criteria_Explanations', 'GPT52_Criteria_Explanations', 'Claude_Response']

First criteria records:
       ID  G

## Step 6: Output Files

Files saved by the verification process:

1. **Datasets/verified/verified_r1_seeds_combined.csv**: Full dialogue rows with columns:
   - All original columns from input CSVs
   - `LLM_used`: Source indicator (Gemini-3.1, GPT-5.2, Gemini-3.1-Claude-4.5, GPT-5.2-Claude-4.5, or Claude-4.5)
   - `Claude_Reasoning`: Claude's explanation for the selection decision

2. **Datasets/verified/verified_r1_criteria_scores.csv**: Detailed per-criterion validation tracking with columns:
   - `ID`: Game/Round identifier
   - `Gemini31_Coherence`, `Gemini31_History`, `Gemini31_Matrix`, `Gemini31_Authenticity`, `Gemini31_Format`: Binary checks (1 = pass, 0 = fail)
   - `Gemini31_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `GPT52_Coherence`, `GPT52_History`, `GPT52_Matrix`, `GPT52_Authenticity`, `GPT52_Format`: Binary checks (1 = pass, 0 = fail)
   - `GPT52_Total`: Overall score (e.g., "5 of 5", "4 of 5")
   - `Selected_LLM`: Which model's output was used
   - `Correction_Mode`: None, Pairwise_Tiebreaker, Targeted, or Full_Custom
   - `Claude_Reasoning`: Claude's explanation for the decision
   - `Gemini31_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)
   - `GPT52_Criteria_Explanations`: JSON with explanations for each criterion (both pass and fail)

**For Paper Analysis:** The criteria CSV enables reporting like: "Gemini-3.1 achieved 95% format compliance (214/225 games)" and identifying which criteria are most challenging for each model. Criteria explanations provide insights into why each criterion passed or failed.

In [10]:
# Save Verified Results
if verified_df is not None and criteria_df is not None:
    # Save combined verified results
    verified_df.to_csv(OUTPUT_FILE_COMBINED, index=False)
    print(f"✓ Verified results saved to {OUTPUT_FILE_COMBINED}")
    print(f"  Total rows: {len(verified_df)}")
    
    # Save criteria tracking with detailed per-criterion scores
    criteria_df.to_csv(OUTPUT_FILE_CRITERIA, index=False)
    print(f"\n✓ Criteria tracking saved to {OUTPUT_FILE_CRITERIA}")
    print(f"  Total records: {len(criteria_df)}")
    
    # Display samples
    print(f"\nSample of verified results (LLM_used distribution):")
    print(verified_df['LLM_used'].value_counts())
    
    print(f"\nSample of criteria records (first 5):")
    print(criteria_df.head(5))
    
    # Display correction mode distribution
    print(f"\nCorrection mode distribution:")
    print(criteria_df['Correction_Mode'].value_counts())
else:
    print("No verified results to save")

✓ Verified results saved to verified_r1_seeds_combined.csv
  Total rows: 225

✓ Criteria tracking saved to verified_r1_criteria_scores.csv
  Total records: 225

Sample of verified results (LLM_used distribution):
LLM_used
GPT-5.2                  158
GPT-5.2-Claude-4.5        31
Gemini-3.1                24
Gemini-3.1-Claude-4.5     10
Claude-4.5                 2
Name: count, dtype: int64

Sample of criteria records (first 5):
       ID  Gemini31_Coherence  Gemini31_History  Gemini31_Matrix  \
0  G026/1                   0                 1                0   
1  G027/1                   0                 1                0   
2  G028/1                   0                 1                1   
3  G029/1                   0                 1                1   
4  G030/1                   1                 1                0   

   Gemini31_Authenticity  Gemini31_Format Gemini31_Total  GPT52_Coherence  \
0                      0                1         2 of 5                1   
1    

## Next Steps: Full Verification

After testing this notebook with the first 2 games and reviewing the results:

1. **Review Sample Results**: Check that binary validation logic works correctly
2. **Check Criteria Distribution**: Examine which criteria pass/fail most frequently
3. **Validate Corrections**: 
   - Check "Targeted" corrections properly fix only the failing criterion
   - Verify "Full_Custom" generations when both models score ≤3/5
   - Confirm "Pairwise_Tiebreaker" selections when both score 5/5
4. **Verify Decision Logic**: Ensure the system correctly handles all scenarios:
   - Both 5/5 → Pairwise comparison
   - One 5/5, other <5 → Selects 5/5
   - At least one 4/5 → Targeted correction (defaults to GPT-5.2 if both 4/5)
   - Both ≤3/5 → Full custom generation
5. **Scale Up**: Change `TEST_LIMIT = 2` to `TEST_LIMIT = len(gemini3_df)` to process all 225 games
6. **Analyze Results**: Use criteria CSV to calculate per-criterion pass rates for your paper
7. **Final Merge**: Combine verified results with 25 manually generated seeds for complete 250-seed dataset

**Important Notes:**
- Current test uses TEST_LIMIT = 2 to validate the system
- Full verification of 225 games will take ~4-5 minutes (1 sec delay + longer max_tokens)
- Per-criterion tracking enables detailed analysis for paper (e.g., "Format: 98% pass rate")

## Comprehensive Analysis: Binary Multi-Criteria Validation Results

Detailed analysis of the verified r1 criteria scores including:
- Per-criterion passing rates for both LLMs
- Average performance metrics
- LLM selection statistics
- Correction mode distribution
- Most/least reliable criteria
- Additional insights

In [ ]:
# ── COMPREHENSIVE ANALYSIS (All Sections Combined) ───────────────────────────
import pandas as pd
import numpy as np

criteria_df = pd.read_csv(VERIFIED_DIR / 'verified_r1_criteria_scores.csv')
criteria_names = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']

# Pre-build score lists used by all later sections
gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]

print("="*80)
print("COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results")
print("="*80)
print(f"\nTotal Records Analyzed: {len(criteria_df)}")

# ── SECTION 1: PER-CRITERION PASSING RATES ───────────────────────────────────
print("\n" + "="*80)
print("1. PER-CRITERION PASSING RATES")
print("="*80)

print("\nGemini-3.1 Passing Rates:")
gemini31_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'Gemini31_{c}'].sum() / len(criteria_df)) * 100
    gemini31_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'Gemini31_{c}'].sum())}/{len(criteria_df)})")

print("\nGPT-5.2 Passing Rates:")
gpt52_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'GPT52_{c}'].sum() / len(criteria_df)) * 100
    gpt52_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'GPT52_{c}'].sum())}/{len(criteria_df)})")

print("\nComparative Analysis (GPT-5.2 vs Gemini-3.1):")
for c in criteria_names:
    diff = gpt52_passing_rates[c] - gemini31_passing_rates[c]
    winner = "GPT-5.2" if diff > 0 else ("Gemini-3.1" if diff < 0 else "TIE")
    print(f"  {c:15s}: {diff:+6.2f}% difference (Winner: {winner})")

# ── SECTION 2: AVERAGE PERFORMANCE METRICS ───────────────────────────────────
gemini31_avg = np.mean(gemini31_scores)
gpt52_avg    = np.mean(gpt52_scores)
gemini31_std = np.std(gemini31_scores)
gpt52_std    = np.std(gpt52_scores)
both_perfect = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("\n" + "="*80)
print("2. AVERAGE PERFORMANCE METRICS")
print("="*80)

print(f"\nGemini-3.1:")
print(f"  Average Score: {gemini31_avg:.3f}/5 (SD: {gemini31_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gemini31_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gemini31_scores))*100:5.2f}%)")

print(f"\nGPT-5.2:")
print(f"  Average Score: {gpt52_avg:.3f}/5 (SD: {gpt52_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gpt52_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gpt52_scores))*100:5.2f}%)")

print(f"\nOverall Comparison:")
if gpt52_avg > gemini31_avg:
    print(f"  GPT-5.2 outperforms Gemini-3.1 by {gpt52_avg - gemini31_avg:.3f} points ({((gpt52_avg - gemini31_avg) / gemini31_avg) * 100:.2f}%)")
elif gemini31_avg > gpt52_avg:
    print(f"  Gemini-3.1 outperforms GPT-5.2 by {gemini31_avg - gpt52_avg:.3f} points")
else:
    print(f"  Both models perform equally: {gemini31_avg:.3f}/5")

# ── SECTION 3: LLM SELECTION STATISTICS ──────────────────────────────────────
selection_counts = criteria_df['Selected_LLM'].value_counts()

print("\n" + "="*80)
print("3. LLM SELECTION STATISTICS")
print("="*80)
print("\nSelection Breakdown:")
for sel, cnt in selection_counts.items():
    print(f"  {sel:30s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

# ── SECTION 4: CORRECTION MODE DISTRIBUTION ──────────────────────────────────
correction_counts = criteria_df['Correction_Mode'].value_counts()
targeted_count    = len(criteria_df[criteria_df['Correction_Mode'] == 'Targeted'])
both_4of5         = sum((g4 == 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gemini31_4of5= sum((g4 == 4 and g5 != 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gpt52_4of5   = sum((g4 != 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("\n" + "="*80)
print("4. CORRECTION MODE DISTRIBUTION")
print("="*80)
print("\nCorrection Mode Breakdown:")
for mode, cnt in correction_counts.items():
    print(f"  {str(mode):20s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

print(f"\nTargeted Correction Breakdown:")
print(f"  Total targeted corrections:         {targeted_count}")
print(f"  Both scored 4/5 (default GPT-5.2):  {both_4of5:4d} ({(both_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only Gemini-3.1 scored 4/5:         {only_gemini31_4of5:4d} ({(only_gemini31_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only GPT-5.2 scored 4/5:            {only_gpt52_4of5:4d} ({(only_gpt52_4of5/len(criteria_df))*100:5.2f}%)")
print(f"\n  Full custom generations: {len(criteria_df[criteria_df['Correction_Mode'] == 'Full_Custom']):4d}")
print(f"  Targeted corrections:    {targeted_count:4d}")

# ── SECTION 5: CRITERIA RANKING ──────────────────────────────────────────────
gemini31_sorted = sorted(gemini31_passing_rates.items(), key=lambda x: x[1])
gpt52_sorted    = sorted(gpt52_passing_rates.items(),    key=lambda x: x[1])

print("\n" + "="*80)
print("5. MOST FAILING AND PASSING CRITERIA")
print("="*80)
print("\nGemini-3.1:")
print(f"  Most Challenging: {gemini31_sorted[0][0]} ({gemini31_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gemini31_sorted[-1][0]} ({gemini31_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gemini31_sorted])}")
print("\nGPT-5.2:")
print(f"  Most Challenging: {gpt52_sorted[0][0]} ({gpt52_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gpt52_sorted[-1][0]} ({gpt52_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gpt52_sorted])}")

# ── SECTION 5a: DETAILED SELECTION BREAKDOWN ─────────────────────────────────
gemini31_direct    = selection_counts.get('Gemini-3.1', 0)
gemini31_corrected = selection_counts.get('Gemini-3.1-Claude-4.5', 0)
gpt52_direct       = selection_counts.get('GPT-5.2', 0)
gpt52_corrected    = selection_counts.get('GPT-5.2-Claude-4.5', 0)
claude_full_custom = selection_counts.get('Claude-4.5', 0)

print("\n" + "="*80)
print("5a. DETAILED SELECTION BREAKDOWN")
print("="*80)
print(f"\n  Gemini-3.1 selected directly:          {gemini31_direct:4d} ({(gemini31_direct/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 corrected by Claude:        {gemini31_corrected:4d} ({(gemini31_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 total usage:                {gemini31_direct + gemini31_corrected:4d} ({((gemini31_direct + gemini31_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  GPT-5.2 selected directly:             {gpt52_direct:4d} ({(gpt52_direct/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 corrected by Claude:           {gpt52_corrected:4d} ({(gpt52_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 total usage:                   {gpt52_direct + gpt52_corrected:4d} ({((gpt52_direct + gpt52_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  Claude full custom (both <=3/5):       {claude_full_custom:4d} ({(claude_full_custom/len(criteria_df))*100:5.2f}%)")
print(f"  Pairwise tiebreaker (both 5/5):        {both_perfect:4d} ({(both_perfect/len(criteria_df))*100:5.2f}%)")

# ── SECTION 6: ADDITIONAL INSIGHTS ───────────────────────────────────────────
one_perfect = sum((g4 == 5) != (g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))
both_low    = sum((g4 <= 3 and g5 <= 3)   for g4, g5 in zip(gemini31_scores, gpt52_scores))
agreement   = sum(g4 == g5                 for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("\n" + "="*80)
print("6. ADDITIONAL INSIGHTS")
print("="*80)
print(f"\nBoth models scored 5/5:       {both_perfect:4d} games ({(both_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Exactly one model scored 5/5: {one_perfect:4d} games ({(one_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Both models scored <=3/5:     {both_low:4d} games ({(both_low/len(criteria_df))*100:5.2f}%)")
print(f"Score Agreement (same total): {agreement:4d} games ({(agreement/len(criteria_df))*100:5.2f}%)")

print("\nCriteria where both models fail together:")
for c in criteria_names:
    both_fail = int(((criteria_df[f'Gemini31_{c}'] == 0) & (criteria_df[f'GPT52_{c}'] == 0)).sum())
    print(f"  {c:15s}: {both_fail:4d} games ({(both_fail/len(criteria_df))*100:5.2f}%)")

print("\nPer-Criterion Agreement (both pass or both fail):")
for c in criteria_names:
    agree_count = int((criteria_df[f'Gemini31_{c}'] == criteria_df[f'GPT52_{c}']).sum())
    print(f"  {c:15s}: {agree_count:4d} agreements ({(agree_count/len(criteria_df))*100:5.2f}%)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results

Total Records Analyzed: 225

1. PER-CRITERION PASSING RATES

Gemini-3.1 Passing Rates:
  Coherence      :  29.33% (66/225)
  History        :  93.78% (211/225)
  Matrix         :  45.33% (102/225)
  Authenticity   :  50.22% (113/225)
  Format         : 100.00% (225/225)

GPT-5.2 Passing Rates:
  Coherence      :  93.78% (211/225)
  History        :  95.11% (214/225)
  Matrix         :  78.22% (176/225)
  Authenticity   :  95.11% (214/225)
  Format         : 100.00% (225/225)

Comparative Analysis (GPT-5.2 vs Gemini-3.1):
  Coherence      : +64.44% difference (Winner: GPT-5.2)
  History        :  +1.33% difference (Winner: GPT-5.2)
  Matrix         : +32.89% difference (Winner: GPT-5.2)
  Authenticity   : +44.89% difference (Winner: GPT-5.2)
  Format         :  +0.00% difference (Winner: TIE)

2. AVERAGE PERFORMANCE METRICS

Gemini-3.1:
  Average Score: 3.187/5 (SD: 1.007)
  Score Distribution:
    0/5:    0 games ( 0.00%

In [1]:
# ── SETUP: Load data and shared variables ────────────────────────────────────
import pandas as pd
import numpy as np

criteria_df = pd.read_csv(VERIFIED_DIR / 'verified_r1_criteria_scores.csv')
criteria_names = ['Coherence', 'History', 'Matrix', 'Authenticity', 'Format']

# Pre-build score lists used by all later sections
gemini31_scores = [sum(row[f'Gemini31_{c}'] for c in criteria_names) for _, row in criteria_df.iterrows()]
gpt52_scores    = [sum(row[f'GPT52_{c}']    for c in criteria_names) for _, row in criteria_df.iterrows()]

print("="*80)
print("COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results")
print("="*80)
print(f"\nTotal Records Analyzed: {len(criteria_df)}")

# ── SECTION 1: PER-CRITERION PASSING RATES ───────────────────────────────────
print("\n" + "="*80)
print("1. PER-CRITERION PASSING RATES")
print("="*80)

print("\nGemini-3.1 Passing Rates:")
gemini31_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'Gemini31_{c}'].sum() / len(criteria_df)) * 100
    gemini31_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'Gemini31_{c}'].sum())}/{len(criteria_df)})")

print("\nGPT-5.2 Passing Rates:")
gpt52_passing_rates = {}
for c in criteria_names:
    r = (criteria_df[f'GPT52_{c}'].sum() / len(criteria_df)) * 100
    gpt52_passing_rates[c] = r
    print(f"  {c:15s}: {r:6.2f}% ({int(criteria_df[f'GPT52_{c}'].sum())}/{len(criteria_df)})")

print("\nComparative Analysis (GPT-5.2 vs Gemini-3.1):")
for c in criteria_names:
    diff = gpt52_passing_rates[c] - gemini31_passing_rates[c]
    winner = "GPT-5.2" if diff > 0 else ("Gemini-3.1" if diff < 0 else "TIE")
    print(f"  {c:15s}: {diff:+6.2f}% difference (Winner: {winner})")

COMPREHENSIVE ANALYSIS: Binary Multi-Criteria Validation Results

Total Records Analyzed: 225

1. PER-CRITERION PASSING RATES

Gemini-3.1 Passing Rates:
  Coherence      :  29.33% (66/225)
  History        :  93.78% (211/225)
  Matrix         :  45.33% (102/225)
  Authenticity   :  50.22% (113/225)
  Format         : 100.00% (225/225)

GPT-5.2 Passing Rates:
  Coherence      :  93.78% (211/225)
  History        :  95.11% (214/225)
  Matrix         :  78.22% (176/225)
  Authenticity   :  95.11% (214/225)
  Format         : 100.00% (225/225)

Comparative Analysis (GPT-5.2 vs Gemini-3.1):
  Coherence      : +64.44% difference (Winner: GPT-5.2)
  History        :  +1.33% difference (Winner: GPT-5.2)
  Matrix         : +32.89% difference (Winner: GPT-5.2)
  Authenticity   : +44.89% difference (Winner: GPT-5.2)
  Format         :  +0.00% difference (Winner: TIE)


In [13]:
# ── SECTION 2: AVERAGE PERFORMANCE METRICS ───────────────────────────────────
gemini31_avg = np.mean(gemini31_scores)
gpt52_avg    = np.mean(gpt52_scores)
gemini31_std = np.std(gemini31_scores)
gpt52_std    = np.std(gpt52_scores)
both_perfect = sum((g4 == 5 and g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("="*80)
print("2. AVERAGE PERFORMANCE METRICS")
print("="*80)

print(f"\nGemini-3.1:")
print(f"  Average Score: {gemini31_avg:.3f}/5 (SD: {gemini31_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gemini31_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gemini31_scores))*100:5.2f}%)")

print(f"\nGPT-5.2:")
print(f"  Average Score: {gpt52_avg:.3f}/5 (SD: {gpt52_std:.3f})")
print(f"  Score Distribution:")
for score in range(6):
    count = gpt52_scores.count(score)
    print(f"    {score}/5: {count:4d} games ({(count/len(gpt52_scores))*100:5.2f}%)")

print(f"\nOverall Comparison:")
if gpt52_avg > gemini31_avg:
    print(f"  GPT-5.2 outperforms Gemini-3.1 by {gpt52_avg - gemini31_avg:.3f} points ({((gpt52_avg - gemini31_avg) / gemini31_avg) * 100:.2f}%)")
elif gemini31_avg > gpt52_avg:
    print(f"  Gemini-3.1 outperforms GPT-5.2 by {gemini31_avg - gpt52_avg:.3f} points")
else:
    print(f"  Both models perform equally: {gemini31_avg:.3f}/5")

2. AVERAGE PERFORMANCE METRICS

Gemini-3.1:
  Average Score: 3.187/5 (SD: 1.007)
  Score Distribution:
    0/5:    0 games ( 0.00%)
    1/5:    1 games ( 0.44%)
    2/5:   65 games (28.89%)
    3/5:   79 games (35.11%)
    4/5:   51 games (22.67%)
    5/5:   29 games (12.89%)

GPT-5.2:
  Average Score: 4.622/5 (SD: 0.670)
  Score Distribution:
    0/5:    0 games ( 0.00%)
    1/5:    0 games ( 0.00%)
    2/5:    4 games ( 1.78%)
    3/5:   12 games ( 5.33%)
    4/5:   49 games (21.78%)
    5/5:  160 games (71.11%)

Overall Comparison:
  GPT-5.2 outperforms Gemini-3.1 by 1.436 points (45.05%)


In [14]:
# ── SECTION 3: LLM SELECTION STATISTICS ──────────────────────────────────────
selection_counts = criteria_df['Selected_LLM'].value_counts()

print("="*80)
print("3. LLM SELECTION STATISTICS")
print("="*80)
print("\nSelection Breakdown:")
for sel, cnt in selection_counts.items():
    print(f"  {sel:30s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

3. LLM SELECTION STATISTICS

Selection Breakdown:
  GPT-5.2                       :  158 (70.22%)
  GPT-5.2-Claude-4.5            :   31 (13.78%)
  Gemini-3.1                    :   24 (10.67%)
  Gemini-3.1-Claude-4.5         :   10 ( 4.44%)
  Claude-4.5                    :    2 ( 0.89%)


In [2]:
# ── SECTION 4: CORRECTION MODE DISTRIBUTION ──────────────────────────────────
correction_counts = criteria_df['Correction_Mode'].value_counts()
targeted_count    = len(criteria_df[criteria_df['Correction_Mode'] == 'Targeted'])
both_4of5         = sum((g4 == 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gemini31_4of5= sum((g4 == 4 and g5 != 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))
only_gpt52_4of5   = sum((g4 != 4 and g5 == 4) for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("="*80)
print("4. CORRECTION MODE DISTRIBUTION")
print("="*80)
print("\nCorrection Mode Breakdown:")
for mode, cnt in correction_counts.items():
    print(f"  {str(mode):20s}: {cnt:4d} ({(cnt/len(criteria_df))*100:5.2f}%)")

print(f"\nTargeted Correction Breakdown:")
print(f"  Total targeted corrections:         {targeted_count}")
print(f"  Both scored 4/5 (pairwise judgement applied):  {both_4of5:4d} ({(both_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only Gemini-3.1 scored 4/5:         {only_gemini31_4of5:4d} ({(only_gemini31_4of5/len(criteria_df))*100:5.2f}%)")
print(f"  Only GPT-5.2 scored 4/5:            {only_gpt52_4of5:4d} ({(only_gpt52_4of5/len(criteria_df))*100:5.2f}%)")
print(f"\n  Full custom generations: {len(criteria_df[criteria_df['Correction_Mode'] == 'Full_Custom']):4d}")
print(f"  Targeted corrections:    {targeted_count:4d}")

4. CORRECTION MODE DISTRIBUTION

Correction Mode Breakdown:
  Targeted            :   41 (18.22%)
  Pairwise_Tiebreaker :    8 ( 3.56%)
  Full_Custom         :    2 ( 0.89%)

Targeted Correction Breakdown:
  Total targeted corrections:         41
  Both scored 4/5 (pairwise judgement applied):    13 ( 5.78%)
  Only Gemini-3.1 scored 4/5:           38 (16.89%)
  Only GPT-5.2 scored 4/5:              36 (16.00%)

  Full custom generations:    2
  Targeted corrections:      41


In [16]:
# ── SECTION 5: CRITERIA RANKING + SECTION 5a: DETAILED SELECTION ─────────────
gemini31_sorted = sorted(gemini31_passing_rates.items(), key=lambda x: x[1])
gpt52_sorted    = sorted(gpt52_passing_rates.items(),    key=lambda x: x[1])

print("="*80)
print("5. MOST FAILING AND PASSING CRITERIA")
print("="*80)
print("\nGemini-3.1:")
print(f"  Most Challenging: {gemini31_sorted[0][0]} ({gemini31_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gemini31_sorted[-1][0]} ({gemini31_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gemini31_sorted])}")
print("\nGPT-5.2:")
print(f"  Most Challenging: {gpt52_sorted[0][0]} ({gpt52_sorted[0][1]:.2f}% pass rate)")
print(f"  Most Reliable:    {gpt52_sorted[-1][0]} ({gpt52_sorted[-1][1]:.2f}% pass rate)")
print(f"  Full Ranking (worst→best): {', '.join([f'{c[0]} ({c[1]:.1f}%)' for c in gpt52_sorted])}")

gemini31_direct    = selection_counts.get('Gemini-3.1', 0)
gemini31_corrected = selection_counts.get('Gemini-3.1-Claude-4.5', 0)
gpt52_direct       = selection_counts.get('GPT-5.2', 0)
gpt52_corrected    = selection_counts.get('GPT-5.2-Claude-4.5', 0)
claude_full_custom = selection_counts.get('Claude-4.5', 0)

print("\n" + "="*80)
print("5a. DETAILED SELECTION BREAKDOWN")
print("="*80)
print(f"\n  Gemini-3.1 selected directly:          {gemini31_direct:4d} ({(gemini31_direct/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 corrected by Claude:        {gemini31_corrected:4d} ({(gemini31_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  Gemini-3.1 total usage:                {gemini31_direct + gemini31_corrected:4d} ({((gemini31_direct + gemini31_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  GPT-5.2 selected directly:             {gpt52_direct:4d} ({(gpt52_direct/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 corrected by Claude:           {gpt52_corrected:4d} ({(gpt52_corrected/len(criteria_df))*100:5.2f}%)")
print(f"  GPT-5.2 total usage:                   {gpt52_direct + gpt52_corrected:4d} ({((gpt52_direct + gpt52_corrected)/len(criteria_df))*100:5.2f}%)")
print()
print(f"  Claude full custom (both <=3/5):       {claude_full_custom:4d} ({(claude_full_custom/len(criteria_df))*100:5.2f}%)")
print(f"  Pairwise tiebreaker (both 5/5):        {both_perfect:4d} ({(both_perfect/len(criteria_df))*100:5.2f}%)")

5. MOST FAILING AND PASSING CRITERIA

Gemini-3.1:
  Most Challenging: Coherence (29.33% pass rate)
  Most Reliable:    Format (100.00% pass rate)
  Full Ranking (worst→best): Coherence (29.3%), Matrix (45.3%), Authenticity (50.2%), History (93.8%), Format (100.0%)

GPT-5.2:
  Most Challenging: Matrix (78.22% pass rate)
  Most Reliable:    Format (100.00% pass rate)
  Full Ranking (worst→best): Matrix (78.2%), Coherence (93.8%), History (95.1%), Authenticity (95.1%), Format (100.0%)

5a. DETAILED SELECTION BREAKDOWN

  Gemini-3.1 selected directly:            24 (10.67%)
  Gemini-3.1 corrected by Claude:          10 ( 4.44%)
  Gemini-3.1 total usage:                  34 (15.11%)

  GPT-5.2 selected directly:              158 (70.22%)
  GPT-5.2 corrected by Claude:             31 (13.78%)
  GPT-5.2 total usage:                    189 (84.00%)

  Claude full custom (both <=3/5):          2 ( 0.89%)
  Pairwise tiebreaker (both 5/5):           7 ( 3.11%)


In [18]:
# ── SECTION 6: ADDITIONAL INSIGHTS ───────────────────────────────────────────
one_perfect = sum((g4 == 5) != (g5 == 5) for g4, g5 in zip(gemini31_scores, gpt52_scores))
both_low    = sum((g4 <= 3 and g5 <= 3)   for g4, g5 in zip(gemini31_scores, gpt52_scores))
agreement   = sum(g4 == g5                 for g4, g5 in zip(gemini31_scores, gpt52_scores))

print("="*80)
print("6. ADDITIONAL INSIGHTS")
print("="*80)
print(f"\nBoth models scored 5/5:       {both_perfect:4d} games ({(both_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Exactly one model scored 5/5: {one_perfect:4d} games ({(one_perfect/len(criteria_df))*100:5.2f}%)")
print(f"Both models scored <=3/5:     {both_low:4d} games ({(both_low/len(criteria_df))*100:5.2f}%)")
print(f"Score Agreement (same total): {agreement:4d} games ({(agreement/len(criteria_df))*100:5.2f}%)")

print("\nCriteria where both models fail together:")
for c in criteria_names:
    both_fail = int(((criteria_df[f'Gemini31_{c}'] == 0) & (criteria_df[f'GPT52_{c}'] == 0)).sum())
    print(f"  {c:15s}: {both_fail:4d} games ({(both_fail/len(criteria_df))*100:5.2f}%)")

print("\nPer-Criterion Agreement (both pass or both fail):")
for c in criteria_names:
    agree_count = int((criteria_df[f'Gemini31_{c}'] == criteria_df[f'GPT52_{c}']).sum())
    print(f"  {c:15s}: {agree_count:4d} agreements ({(agree_count/len(criteria_df))*100:5.2f}%)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

6. ADDITIONAL INSIGHTS

Both models scored 5/5:          7 games ( 3.11%)
Exactly one model scored 5/5:  175 games (77.78%)
Both models scored <=3/5:        2 games ( 0.89%)
Score Agreement (same total):   22 games ( 9.78%)

Criteria where both models fail together:
  Coherence      :    3 games ( 1.33%)
  History        :    3 games ( 1.33%)
  Matrix         :   17 games ( 7.56%)
  Authenticity   :    1 games ( 0.44%)
  Format         :    0 games ( 0.00%)

Per-Criterion Agreement (both pass or both fail):
  Coherence      :   58 agreements (25.78%)
  History        :  206 agreements (91.56%)
  Matrix         :   87 agreements (38.67%)
  Authenticity   :  104 agreements (46.22%)
  Format         :  225 agreements (100.00%)

ANALYSIS COMPLETE
